# Partie 2 — Classification avec CamemBERT
**Projet Algorithmique Python S2 — Master 1 IdL 2025/2026**

Modele : **CamemBERT** (camembert-base) via HuggingFace Transformers

Tache : classification supervisee 5 categories

> Executer sur **Google Colab** (GPU T4 gratuit) + uploader `articles.csv`

| Etape | Structure | Type |
|-------|-----------|------|
| Chargement CSV | DataFrame | pandas.DataFrame |
| Labels encodes | Vecteur entiers | list[int] |
| Tokens | Dict tenseurs | {input_ids, attention_mask} |
| Sortie modele | Logits | torch.Tensor(batch, 5) |
| Prediction | Label predit | int -> str |

In [ ]:
!pip install transformers datasets torch scikit-learn pandas --quiet

In [ ]:
import pandas as pd
import torch
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    CamembertTokenizer,
    CamembertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from torch.utils.data import Dataset

print('GPU disponible :', torch.cuda.is_available())
print('Librairies importees.')

In [ ]:
# Chargement du dataset
# Structure : DataFrame pandas, 50 lignes, 6 colonnes
df = pd.read_csv('articles.csv')
df = df.dropna(subset=['texte', 'label'])

# Combiner titre + texte
df['texte_complet'] = df['titre'].fillna('') + ' ' + df['texte'].fillna('')

print('Articles : ' + str(len(df)))
print(df['label'].value_counts())

In [ ]:
# Encodage des labels
# Structure : vecteur d'entiers
# economie=0, politique=1, sante=2, sciences=3, sport=4
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['label'])

print('Mapping :')
for i, c in enumerate(le.classes_):
    print('  ' + c + ' -> ' + str(i))

NUM_LABELS = len(le.classes_)

In [ ]:
# Split 80/20
X = df['texte_complet'].tolist()
y = df['label_id'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train : ' + str(len(X_train)) + ' | Test : ' + str(len(X_test)))

In [ ]:
# Tokenisation CamemBERT
# Structure apres tokenisation :
#   input_ids     : Tensor(1, 128) - indices tokens
#   attention_mask : Tensor(1, 128) - 1=token, 0=padding
MODEL_NAME = 'camembert-base'
tokenizer = CamembertTokenizer.from_pretrained(MODEL_NAME)

exemple = tokenizer(X_train[0], return_tensors='pt', truncation=True, max_length=128)
print('input_ids shape     : ' + str(exemple['input_ids'].shape))
print('attention_mask shape : ' + str(exemple['attention_mask'].shape))

In [ ]:
class ArticleDataset(Dataset):
    """
    Dataset PyTorch.
    Structure : dict {input_ids, attention_mask, labels}
    """
    def __init__(self, textes, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            textes, padding=True, truncation=True,
            max_length=max_length, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = ArticleDataset(X_train, y_train, tokenizer)
test_dataset  = ArticleDataset(X_test,  y_test,  tokenizer)
print('Datasets crees. Train=' + str(len(train_dataset)) + ' Test=' + str(len(test_dataset)))

In [ ]:
# Chargement CamemBERT + tete classification
# Sortie : logits Tensor(batch_size, NUM_LABELS)
model = CamembertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS
)
print('Modele charge : ' + MODEL_NAME + ' (' + str(NUM_LABELS) + ' classes)')

In [ ]:
def compute_metrics(eval_pred):
    """
    Calcule accuracy et F1.
    logits Tensor(batch, 5) -> argmax -> predictions
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions, average='weighted')
    }

training_args = TrainingArguments(
    output_dir='/tmp/resultats',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    save_strategy='no',
    logging_strategy='epoch',
    load_best_model_at_end=False,
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)
print('Trainer configure.')

In [ ]:
# Entrainement (~5-10 min sur Colab GPU T4)
print('Debut entrainement...')
trainer.train()
print('Termine !')

In [ ]:
# Evaluation
resultats = trainer.evaluate()
print('Accuracy : ' + str(round(resultats['eval_accuracy'] * 100, 2)) + '%')
print('F1-score : ' + str(round(resultats['eval_f1'] * 100, 2)) + '%')

: 

In [ ]:
# Rapport de classification
# logits : Tensor(nb_exemples, 5) -> argmax -> predictions
preds_output = trainer.predict(test_dataset)
logits = preds_output.predictions
y_pred = np.argmax(logits, axis=1)
y_true = preds_output.label_ids

print('Structure logits : ' + str(logits.shape))
print(classification_report(y_true, y_pred, target_names=le.classes_))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Calcul de la matrice de confusion
cm = confusion_matrix(y_true, y_pred)

# Affichage
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('Matrice de confusion — CamemBERT', fontsize=14)
plt.ylabel('Label reel')
plt.xlabel('Label predit')
plt.tight_layout()
plt.savefig('matrice_confusion.png', dpi=150)
plt.show()
print('Matrice sauvegardee : matrice_confusion.png')

In [ ]:
def predire(texte):
    """
    Predit la categorie d'un texte.
    Pipeline : texte -> tokens -> logits Tensor(1,5) -> argmax -> label
    """
    model.eval()
    tokens = tokenizer(texte, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        sortie = model(**tokens)
        indice = sortie.logits.argmax().item()
    return le.inverse_transform([indice])[0]

tests = [
    'Le gouvernement presente son budget a lAssemblee nationale.',
    'Le PSG remporte le match contre Lyon 3 a 1 en Ligue 1.',
    'Une nouvelle etude montre les effets du vaccin contre la grippe.',
    'Les chercheurs ont decouvert une nouvelle espece de bacterie.',
    'L inflation ralentit en France mais reste elevee.',
]

for texte in tests:
    print(texte[:60] + ' => ' + predire(texte))